# Sesión 14 — Transformers II: Construyendo el Modelo Completo

En esta sesión **construiremos un transformer completo** paso a paso:
- Revisión rápida de la atención
- **Multi-head attention** (implementación manual)
- **Positional encoding** (sinusoidal)
- **Feedforward network**
- **Layer Normalization** y conexiones residuales
- **Bloque Encoder** completo
- **Máscara causal** para decoder
- **Ejemplo práctico:** Predicción de secuencias (character-level)

Objetivo: entender cómo se ensambla todo el sistema transformer.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)

## 1) Revisión: Scaled Dot-Product Attention

$$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch, seq_len, d_k)
    mask: (batch, seq_len, seq_len) o None
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

# Prueba rápida
batch_size, seq_len, d_model = 2, 4, 8
X = torch.randn(batch_size, seq_len, d_model)
out, w = scaled_dot_product_attention(X, X, X)
print(f"Output shape: {out.shape}, Weights shape: {w.shape}")

## 2) Multi-Head Attention (Manual)

En lugar de una sola atención, dividimos el embedding en **h cabezas** (heads):
- Cada cabeza trabaja en un subespacio de dimensión $d_k = d_{model} / h$
- Concatenamos las salidas y proyectamos

$$
\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O
$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Proyecciones lineales para Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def split_heads(self, x):
        """Divide última dim en (num_heads, d_k)"""
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        
        # 1) Proyectar Q, K, V
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)
        
        # 2) Dividir en cabezas: (batch, num_heads, seq_len, d_k)
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 3) Atención por cabeza
        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask)
        # attn_out: (batch, num_heads, seq_len, d_k)
        
        # 4) Concatenar cabezas
        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(batch_size, -1, self.d_model)
        
        # 5) Proyección de salida
        output = self.W_o(attn_out)
        return output

# Prueba
d_model, num_heads = 16, 4
mha = MultiHeadAttention(d_model, num_heads)
X = torch.randn(2, 5, d_model)
out = mha(X, X, X)
print(f"MHA output shape: {out.shape}")

## 3) Positional Encoding (Sinusoidal)

La atención es **invariante al orden**. Necesitamos agregar información de posición.

$$
PE_{(pos, 2i)} = \sin\left(pos / 10000^{2i/d_{model}}\right)
$$
$$
PE_{(pos, 2i+1)} = \cos\left(pos / 10000^{2i/d_{model}}\right)
$$

In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

# Visualización
pe = positional_encoding(50, 128)
plt.figure(figsize=(10, 4))
plt.imshow(pe.T, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Embedding dim')
plt.title('Positional Encoding')
plt.show()

## 4) Feedforward Network (FFN)

Después de la atención, cada posición pasa por una MLP independiente:

$$
\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Prueba
ffn = FeedForward(d_model=16, d_ff=64)
x = torch.randn(2, 5, 16)
out = ffn(x)
print(f"FFN output shape: {out.shape}")

## 5) Layer Normalization y Conexiones Residuales

Cada subcapa se envuelve con:
$$
\text{LayerNorm}(x + \text{Sublayer}(x))
$$

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention + residual + norm
        attn_out = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # FFN + residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

# Prueba
encoder_layer = TransformerEncoderLayer(d_model=16, num_heads=4, d_ff=64)
x = torch.randn(2, 5, 16)
out = encoder_layer(x)
print(f"Encoder layer output: {out.shape}")

## 6) Máscara Causal para Decoder

En un decoder autoregresivo, el token $i$ **no puede mirar** tokens futuros $j > i$.
Creamos una máscara triangular superior:

In [ ]:
def create_causal_mask(seq_len):
    """Máscara triangular inferior (1 = permitido, 0 = bloqueado)"""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# Visualización
mask = create_causal_mask(6)
plt.imshow(mask, cmap='Blues')
plt.title('Causal Mask (1=allowed, 0=masked)')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.colorbar()
plt.show()

## 7) Transformer Completo (Encoder + Positional Encoding)

Juntamos todo:

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=500, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.register_buffer('pe', positional_encoding(max_len, d_model))
        
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        seq_len = x.size(1)
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = x + self.pe[:seq_len, :]
        x = self.dropout(x)
        
        for layer in self.layers:
            x = layer(x, mask)
        return x

# Prueba
model = TransformerEncoder(vocab_size=100, d_model=64, num_heads=4, d_ff=256, num_layers=2)
tokens = torch.randint(0, 100, (2, 10))
out = model(tokens)
print(f"Transformer output: {out.shape}")

## 8) Ejemplo Práctico: Character-Level Next Token Prediction

Entrenaremos un transformer pequeño para predecir el siguiente caracter en una secuencia.

In [ ]:
# Dataset simple: secuencia de caracteres
text = "hola mundo! este es un ejemplo simple para entrenar un transformer pequeño. " * 100

# Vocabulario
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Vocabulario: {chars}")
print(f"Tamaño: {vocab_size}")

# Convertir a índices
data = torch.tensor([char_to_idx[ch] for ch in text], dtype=torch.long)
print(f"\nData shape: {data.shape}")

In [ ]:
# Dataset simple
class CharDataset(torch.utils.data.Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
    
    def __len__(self):
        return len(self.data) - self.seq_len
    
    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return x, y

seq_len = 32
dataset = CharDataset(data, seq_len)
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Dataset size: {len(dataset)}")
x_sample, y_sample = dataset[0]
print(f"Sample X: {''.join([idx_to_char[i.item()] for i in x_sample])}")
print(f"Sample Y: {''.join([idx_to_char[i.item()] for i in y_sample])}")

In [ ]:
# Modelo para predicción de siguiente token
class CharTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers, dropout=dropout)
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        # Máscara causal
        seq_len = x.size(1)
        mask = create_causal_mask(seq_len).to(x.device)
        
        # Encoder
        x = self.encoder(x, mask)
        
        # Predicción
        logits = self.fc_out(x)
        return logits

model = CharTransformer(vocab_size=vocab_size, d_model=64, num_heads=4, d_ff=128, num_layers=2)
print(f"Número de parámetros: {sum(p.numel() for p in model.parameters())/1e3:.1f}K")

In [ ]:
# Entrenamiento
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

model.train()
losses = []

for epoch in range(50):
    epoch_loss = 0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        
        logits = model(x_batch)
        loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 - Loss: {avg_loss:.4f}")

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
# Generación de texto
def generate(model, start_text, max_len=100, temperature=1.0):
    model.eval()
    context = torch.tensor([char_to_idx[ch] for ch in start_text], dtype=torch.long).unsqueeze(0)
    
    generated = start_text
    
    with torch.no_grad():
        for _ in range(max_len):
            logits = model(context)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()
            next_char = idx_to_char[next_idx]
            generated += next_char
            
            context = torch.cat([context, torch.tensor([[next_idx]])], dim=1)
            if context.size(1) > seq_len:
                context = context[:, -seq_len:]
    
    return generated

# Prueba
start = "hola "
generated_text = generate(model, start, max_len=100, temperature=0.8)
print(f"Texto generado:\n{generated_text}")

## Resumen de la Sesión

Hemos construido:
1. ✅ Multi-head attention (desde cero)
2. ✅ Positional encoding (sinusoidal)
3. ✅ Feedforward network
4. ✅ Layer normalization + residuales
5. ✅ Bloque encoder completo
6. ✅ Máscara causal para decoder
7. ✅ Modelo entrenado en tarea real

**Puntos clave:**
- Los transformers son **modulares**: attention + FFN + norm + residual
- El **positional encoding** es crítico (sin él, el modelo es invariante al orden)
- La **máscara causal** permite entrenar modelos autoregresivos
- Escalan bien porque **todo es paralelizable** (no hay recurrencia)